<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L30_Project_Document_QA_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L30: Project - Document QA System

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 30 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Build an end-to-end document QA system using RAG (L28) and vector search (L29)
- Ingest documents, chunk them, embed, and index
- Answer questions by retrieving context and generating with an LM
- Meet requirements for a production-ready QA pipeline

---

## Concept: Document QA Pipeline

**Document QA** = (1) Ingest: load PDFs/text; (2) Chunk: split into passages; (3) Embed & index (e.g. FAISS); (4) Query: embed question, retrieve top-k chunks; (5) Generate: prompt LM with context + question, return answer. Integrates L16–L29 (transfer learning, RAG, vector DB, efficient inference).

---

In [ ]:
!pip install transformers torch sentence-transformers faiss-cpu -q
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import faiss
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Document Ingestion and Chunking

---

In [ ]:
def simple_chunk(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

corpus = """
Large language models are trained on vast amounts of text. They can answer questions, summarize text, and generate content.
Retrieval-augmented generation (RAG) combines retrieval with generation. You first retrieve relevant documents, then the model generates an answer using that context.
Vector databases store embeddings and allow fast similarity search. FAISS is a popular library for this. Embeddings are dense vectors representing meaning.
"""

chunks = simple_chunk(corpus.strip())
print(f"Chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  {i+1}: {c[:60]}...")

## Part 2: Embed and Index

---

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print("Index ready.")

## Part 3: Retrieve and Generate Answer

---

In [ ]:
def retrieve(query, index, chunks, encoder, top_k=2):
    q = np.array(encoder.encode([query])).astype("float32")
    _, indices = index.search(q, top_k)
    return [chunks[i] for i in indices[0]]

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def answer_question(query, index, chunks, encoder, generator, top_k=2, max_new=40):
    context = retrieve(query, index, chunks, encoder, top_k)
    context_str = " ".join(context)
    prompt = f"Context: {context_str}\n\nQuestion: {query}\nAnswer:"
    out = generator(prompt, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return out[0]["generated_text"][len(prompt):].strip()

q = "What is RAG?"
ans = answer_question(q, index, chunks, encoder, generator)
print(f"Q: {q}")
print(f"A: {ans}")

## Part 4: Run Multiple Questions

---

In [ ]:
questions = ["What is RAG?", "What are vector databases used for?", "What are embeddings?"]
for q in questions:
    a = answer_question(q, index, chunks, encoder, generator)
    print(f"Q: {q}\nA: {a}\n")

## Project Requirements Checklist

- [x] Document ingestion (text/chunks)
- [x] Embedding and vector index (FAISS)
- [x] Retrieval by query
- [x] Generation with context (RAG)
- [ ] Add PDF ingestion (e.g. PyMuPDF) for real docs
- [ ] Add re-ranking or more chunks for better accuracy
- [ ] Deploy as API (e.g. FastAPI) for production

---

## Exercises

1. Load a multi-page PDF, chunk by page or paragraph, and run QA.
2. Compare top_k=1 vs 3 vs 5 on answer quality.
3. Add a simple evaluation: ask 10 questions with known answers and compute exact match or F1.
4. Wrap the pipeline in FastAPI: POST /query with {"question": "..."} returning {"answer": "..."}.

---

## Key Takeaways

1. Document QA = ingest → chunk → embed → index → retrieve → generate.
2. Chunk size and overlap affect retrieval quality; tune for your corpus.
3. Production: add auth, rate limits, logging, and optional re-ranker.

---

## Next Steps

You have completed **Level 2 (Intermediate)**. Proceed to **Level 3 (Advanced)**: training from scratch, distributed training, Flash Attention, long context, and the domain-specific model project (L31–L45).

---

**Congratulations! You built a document QA system.**

---